In [1]:
import json
import os
import pandas as pd
import numpy as np

notebook_path = '/content/Untitled5 (1).ipynb'
if os.path.exists(notebook_path):
    with open(notebook_path, 'r', encoding='utf-8') as f:
        nb_content = json.load(f)
    print(f"Successfully loaded notebook. Total cells: {len(nb_content.get('cells', []))}")
else:
    print(f"Notebook file not found at {notebook_path}. Please make sure the path is correct.")

Successfully loaded notebook. Total cells: 123


In [2]:
for idx, cell in enumerate(nb_content.get('cells', [])):
    source = ''.join(cell.get('source', []))
    if any(keyword in source for keyword in ['FDA', 'CMS', 'spending', 'exclusivity', 'expiry', 'model', 'MAE', 'Scenario', 'README']):
        print(f"Cell {idx} (ID: {cell.get('id', 'N/A')}): {source[:120].strip()}...")

Cell 1 (ID: N/A): **Reasoning**:
We need to inspect the FDA text files, check the CMS datasets (Excel and ZIP archives), and print a detai...
Cell 2 (ID: N/A): import os
import zipfile
import pandas as pd
import openpyxl

def inspect_fda_files():
    print("=== FDA Text Files Aud...
Cell 3 (ID: N/A): **Reasoning**:
We need to read the actual data sheets of representative CMS Excel files to inspect key identifier column...
Cell 4 (ID: N/A): import pandas as pd
import numpy as np

def inspect_cms_spending_sheets():
    print("=== CMS Actual Spending Sheets Aud...
Cell 5 (ID: N/A): **Reasoning**:
We will begin Phase 2 by programmatically extracting all ZIP archives containing CMS Medicare Part D xlsx...
Cell 6 (ID: N/A): import os
import zipfile

def extract_cms_zips():
    print("=== Extracting CMS ZIP Archives ===")
    zips = [f for f i...
Cell 7 (ID: N/A): **Reasoning**:
We need to dynamically discover the column names, determine the correct row headers, and parse all 8 extr...
Cell 8

In [3]:
import os
print("=== Files in /content/ ===")
for f in os.listdir('/content/'):
    print(f, os.path.getsize(os.path.join('/content/', f)) if os.path.isfile(os.path.join('/content/', f)) else '(directory)')

=== Files in /content/ ===
.config (directory)
Untitled5 (1).ipynb 1352005
sample_data (directory)


In [4]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
from difflib import get_close_matches
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- 1. Load and Parse FDA Patent/Exclusivity Data ---
def load_fda_data():
    patent_path = '/content/patent.txt'
    excl_path = '/content/exclusivity.txt'
    prod_path = '/content/products.txt'

    # Load raw text databases
    patent_df = pd.read_csv(patent_path, sep='\t', error_bad_lines=False, warn_bad_lines=False, encoding='latin1')
    excl_df = pd.read_csv(excl_path, sep='\t', error_bad_lines=False, warn_bad_lines=False, encoding='latin1')
    prod_df = pd.read_csv(prod_path, sep='\t', error_bad_lines=False, warn_bad_lines=False, encoding='latin1')

    # Normalize columns
    patent_df.columns = patent_df.columns.str.strip().str.upper()
    excl_df.columns = excl_df.columns.str.strip().str.upper()
    prod_df.columns = prod_df.columns.str.strip().str.upper()

    # Correct Date Parsing & Sorting
    patent_df['PATENT_EXPIRE_DATE'] = pd.to_datetime(patent_df['PATENT_EXPIRE_DATE'], errors='coerce')
    excl_df['EXCLUSIVITY_DATE'] = pd.to_datetime(excl_df['EXCLUSIVITY_DATE'], errors='coerce')

    valid_patents = patent_df['PATENT_EXPIRE_DATE'].notnull().sum()
    valid_excl = excl_df['EXCLUSIVITY_DATE'].notnull().sum()

    # Deduplicate patents and exclusivity
    pat_dedup = patent_df.dropna(subset=['PATENT_EXPIRE_DATE']).sort_values('PATENT_EXPIRE_DATE').groupby('APPL_NO').first().reset_index()
    excl_dedup = excl_df.dropna(subset=['EXCLUSIVITY_DATE']).sort_values('EXCLUSIVITY_DATE').groupby('APPL_NO').first().reset_index()

    fda_merged = pd.merge(prod_df, pat_dedup[['APPL_NO', 'PATENT_EXPIRE_DATE']], on='APPL_NO', how='left')
    fda_merged = pd.merge(fda_merged, excl_dedup[['APPL_NO', 'EXCLUSIVITY_DATE']], on='APPL_NO', how='left')

    return fda_merged, valid_patents, valid_excl

fda_df, valid_patents, valid_excl = load_fda_data()
print(f"FDA records loaded. Valid patent dates: {valid_patents}, Valid exclusivity dates: {valid_excl}")

# --- 2. Load CMS Medicare Part D Spending Data ---
def extract_cms_spending():
    zips = sorted([f for f in os.listdir('/content/') if f.startswith('Medicare Part D Spending by Drug DYT') and f.endswith('.zip')])
    all_data = []

    for z_file in zips:
        year_match = re.search(r'DYT(\d{4})', z_file)
        if not year_match: continue
        year = int(year_match.group(1))

        with zipfile.ZipFile(os.path.join('/content/', z_file), 'r') as z:
            for name in z.namelist():
                if name.endswith('.xlsx') or name.endswith('.xls'):
                    z.extract(name, '/tmp')
                    df_excel = pd.read_excel(os.path.join('/tmp', name))
                    df_excel.columns = df_excel.columns.str.strip().str.upper()

                    # Ensure robust column selection
                    spending_col = None
                    for col in df_excel.columns:
                        if 'SPNDNG' in col or 'SPENDING' in col:
                            if 'TOT' in col or 'TOTAL' in col:
                                spending_col = col
                                break
                    if not spending_col:
                        spending_col = [col for col in df_excel.columns if 'SPNDNG' in col or 'SPENDING' in col][0]

                    # Track unique product name identifier
                    name_col = [col for col in df_excel.columns if 'NAME' in col or 'BRAND' in col][0]

                    temp_df = df_excel[[name_col, spending_col]].copy()
                    temp_df.columns = ['DRUG_NAME', 'TOTAL_SPENDING']
                    temp_df['YEAR'] = year
                    all_data.append(temp_df)

    return pd.concat(all_data, ignore_index=True)

cms_raw = extract_cms_spending()
print(f"CMS data loaded. Row count: {len(cms_raw)}, unique years: {cms_raw['YEAR'].unique()}")


TypeError: read_csv() got an unexpected keyword argument 'error_bad_lines'. Did you mean 'on_bad_lines'?

In [5]:
# --- 3. Exact and Fuzzy Match Coverage Analysis ---
def match_products(cms, fda):
    fda_names = fda['DRUG_NAME'].dropna().unique().tolist()
    cms_names = cms['DRUG_NAME'].dropna().unique()

    matches = {}
    exact_brand_cnt = 0
    exact_generic_cnt = 0
    fuzzy_cnt = 0
    unmatched_cnt = 0

    for name in cms_names:
        cleaned_cms = str(name).strip().upper()
        # Exact match check
        if cleaned_cms in fda_names:
            matches[name] = cleaned_cms
            exact_brand_cnt += 1
        else:
            # Try fuzzy match
            close = get_close_matches(cleaned_cms, fda_names, n=1, cutoff=0.85)
            if close:
                matches[name] = close[0]
                fuzzy_cnt += 1
            else:
                unmatched_cnt += 1

    print(f"Match Counts -> Exact: {exact_brand_cnt}, Fuzzy: {fuzzy_cnt}, Unmatched: {unmatched_cnt}")
    return matches

matches_dict = match_products(cms_raw, fda_df)


NameError: name 'cms_raw' is not defined

In [6]:
# --- 4. Dataset Rebuild, Temporal Modeling, and Scenario Simulation ---
# Standardize matching, calculate chronological lags, train models, and execute scenarios
cms_raw['MATCHED_FDA_NAME'] = cms_raw['DRUG_NAME'].map(matches_dict)
merged_df = pd.merge(cms_raw, fda_df, left_on='MATCHED_FDA_NAME', right_on='DRUG_NAME', how='left', suffixes=('_CMS', '_FDA'))

# Group and calculate lags properly per product-year
merged_df['PREV_YEAR_SPENDING'] = merged_df.groupby('DRUG_NAME_CMS')['TOTAL_SPENDING'].shift(1)
merged_df['PREV_YEAR_SPENDING'] = merged_df['PREV_YEAR_SPENDING'].fillna(0) # Distinguish historical zeros

# Temporal Splitting & Model Training
train_set = merged_df[merged_df['YEAR'] < 2023]
test_set = merged_df[merged_df['YEAR'] == 2023]

print(f"Train observations: {len(train_set)}, Test observations: {len(test_set)}")


NameError: name 'cms_raw' is not defined

In [7]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
from difflib import get_close_matches
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- 1. Load and Parse FDA Patent/Exclusivity Data ---
def load_fda_data():
    patent_path = '/content/patent.txt'
    excl_path = '/content/exclusivity.txt'
    prod_path = '/content/products.txt'

    # Load raw text databases with modern error handling
    patent_df = pd.read_csv(patent_path, sep='\t', on_bad_lines='skip', encoding='latin1')
    excl_df = pd.read_csv(excl_path, sep='\t', on_bad_lines='skip', encoding='latin1')
    prod_df = pd.read_csv(prod_path, sep='\t', on_bad_lines='skip', encoding='latin1')

    # Normalize columns
    for df in [patent_df, excl_df, prod_df]:
        df.columns = df.columns.str.strip().str.upper()

    # Parse patent and exclusivity dates
    patent_df['PATENT_EXPIRE_DATE'] = pd.to_datetime(patent_df['PATENT_EXPIRE_DATE'], errors='coerce')
    excl_df['EXCLUSIVITY_DATE'] = pd.to_datetime(excl_df['EXCLUSIVITY_DATE'], errors='coerce')

    # Deduplicate patents and exclusivity
    pat_dedup = patent_df.dropna(subset=['PATENT_EXPIRE_DATE']).sort_values('PATENT_EXPIRE_DATE').groupby('APPL_NO').first().reset_index()
    excl_dedup = excl_df.dropna(subset=['EXCLUSIVITY_DATE']).sort_values('EXCLUSIVITY_DATE').groupby('APPL_NO').first().reset_index()

    fda_merged = pd.merge(prod_df, pat_dedup[['APPL_NO', 'PATENT_EXPIRE_DATE']], on='APPL_NO', how='left')
    fda_merged = pd.merge(fda_merged, excl_dedup[['APPL_NO', 'EXCLUSIVITY_DATE']], on='APPL_NO', how='left')

    return fda_merged, patent_df, excl_df

fda_df, patent_df, excl_df = load_fda_data()
print(f"FDA records loaded. Valid patent dates: {patent_df['PATENT_EXPIRE_DATE'].notnull().sum()}, Valid exclusivity dates: {excl_df['EXCLUSIVITY_DATE'].notnull().sum()}")


KeyError: 'PATENT_EXPIRE_DATE'

In [10]:
import pandas as pd

for f_name in ['patent.txt', 'exclusivity.txt', 'products.txt']:
    path = f'/content/{f_name}'
    df_sample = pd.read_csv(path, sep='\t', nrows=5, encoding='latin1')
    print(f"\n=== File: {f_name} ===")
    print("Original Columns:", list(df_sample.columns))
    print(df_sample.head(2))


=== File: patent.txt ===
Original Columns: ['Appl_Type~Appl_No~Product_No~Patent_No~Patent_Expire_Date_Text~Drug_Substance_Flag~Drug_Product_Flag~Patent_Use_Code~Delist_Flag~Submission_Date']
  Appl_Type~Appl_No~Product_No~Patent_No~Patent_Expire_Date_Text~Drug_Substance_Flag~Drug_Product_Flag~Patent_Use_Code~Delist_Flag~Submission_Date
0        N~020610~001~7625884~Aug 24, 2026~~~U-141~~                                                                                              
1       N~018613~001~7560445~Feb 1, 2027~Y~Y~U-986~~                                                                                              

=== File: exclusivity.txt ===
Original Columns: ['Appl_Type~Appl_No~Product_No~Exclusivity_Code~Exclusivity_Date']
  Appl_Type~Appl_No~Product_No~Exclusivity_Code~Exclusivity_Date
0                      N~017031~001~RTO~Jul 13, 2026            
1                    N~018680~001~D-193~Jun 28, 2027            

=== File: products.txt ===
Original Columns: ['Ingredi

In [11]:
import pandas as pd

for f_name in ['patent.txt', 'exclusivity.txt', 'products.txt']:
    path = f'/content/{f_name}'
    df_sample = pd.read_csv(path, sep='\t', nrows=5, encoding='latin1')
    print(f"\n=== File: {f_name} ===")
    print("Original Columns:", list(df_sample.columns))
    print(df_sample.head(2))


=== File: patent.txt ===
Original Columns: ['Appl_Type~Appl_No~Product_No~Patent_No~Patent_Expire_Date_Text~Drug_Substance_Flag~Drug_Product_Flag~Patent_Use_Code~Delist_Flag~Submission_Date']
  Appl_Type~Appl_No~Product_No~Patent_No~Patent_Expire_Date_Text~Drug_Substance_Flag~Drug_Product_Flag~Patent_Use_Code~Delist_Flag~Submission_Date
0        N~020610~001~7625884~Aug 24, 2026~~~U-141~~                                                                                              
1       N~018613~001~7560445~Feb 1, 2027~Y~Y~U-986~~                                                                                              

=== File: exclusivity.txt ===
Original Columns: ['Appl_Type~Appl_No~Product_No~Exclusivity_Code~Exclusivity_Date']
  Appl_Type~Appl_No~Product_No~Exclusivity_Code~Exclusivity_Date
0                      N~017031~001~RTO~Jul 13, 2026            
1                    N~018680~001~D-193~Jun 28, 2027            

=== File: products.txt ===
Original Columns: ['Ingredi

In [42]:
# --- 2. Load CMS Medicare Part D Spending Data ---
def extract_cms_spending():
    # Target direct unzipped Excel worksheets in /content/ directory, strictly excluding backups and copy duplicates
    all_files = os.listdir('/content/')
    files = []
    for f in all_files:
        if f.startswith('DSD_PTD_') and f.endswith('.xlsx'):
            if 'Methodology' not in f and 'Copy' not in f and '(' not in f:
                files.append(f)
    files = sorted(list(set(files)))

    all_data = []

    for f_file in files:
        # Extract year from filename
        year_match = re.search(r'DYT(\d{2})|DYT(\d{4})', f_file)
        if not year_match:
            continue
        groups = [g for g in year_match.groups() if g is not None]
        dyt_val = int(groups[0])
        if dyt_val > 100:
            year = dyt_val
        else:
            year = 2000 + dyt_val

        filepath = os.path.join('/content/', f_file)
        xl = pd.ExcelFile(filepath)

        # Locate correct spending sheet
        sheet_name = None
        for s in xl.sheet_names:
            s_upper = s.upper()
            if 'SPENDING' in s_upper and 'UTILIZATION' in s_upper:
                sheet_name = s
                break
        if not sheet_name:
            for s in xl.sheet_names:
                s_upper = s.upper()
                if any(k in s_upper for k in ['DATA', 'SPENDING', 'DRUG', 'PRODUCT']) and 'INDEX' not in s_upper and 'GUIDE' not in s_upper:
                    sheet_name = s
                    break
        if not sheet_name:
            sheet_name = xl.sheet_names[0]

        # Read first 15 rows of the sheet to find the header row containing 'BRAND NAME' or 'GENERIC NAME'
        df_sample = pd.read_excel(filepath, sheet_name=sheet_name, nrows=15, header=None)
        header_row = 0
        for idx in range(len(df_sample)):
            row_vals = [str(x).strip().upper() for x in df_sample.iloc[idx].values if pd.notnull(x)]
            if any('BRAND NAME' in r or 'GENERIC NAME' in r or 'DRUG NAME' in r for r in row_vals):
                header_row = idx
                break

        # Load worksheet with the detected header row
        df_excel = pd.read_excel(filepath, sheet_name=sheet_name, skiprows=header_row)
        df_excel.columns = df_excel.columns.str.strip().str.upper()

        # Identify the Drug Name and Total Spending columns
        name_col = None
        for col in df_excel.columns:
            col_upper = str(col).upper()
            if 'BRAND NAME' in col_upper or 'DRUG NAME' in col_upper or col_upper == 'BRAND_NAME':
                name_col = col
                break
        if not name_col:
            for col in df_excel.columns:
                col_upper = str(col).upper()
                if 'GENERIC NAME' in col_upper or col_upper == 'GENERIC_NAME':
                    name_col = col
                    break

        spending_col = None
        for col in df_excel.columns:
            col_upper = str(col).upper()
            if 'TOTAL SPENDING' in col_upper or 'TOTAL_SPENDING' in col_upper or 'TOT_SPNDNG' in col_upper or 'TOTAL SPNDNG' in col_upper:
                spending_col = col
                break
        if not spending_col:
            for col in df_excel.columns:
                col_upper = str(col).upper()
                if 'SPENDING' in col_upper or 'SPNDNG' in col_upper:
                    if 'AVERAGE' not in col_upper and 'ANNUAL' not in col_upper and 'CHANGE' not in col_upper:
                        spending_col = col
                        break

        print(f"Year: {year} | File: {f_file} | Identified Columns: Name: {name_col}, Spending: {spending_col}")

        if name_col and spending_col:
            temp_df = df_excel[[name_col, spending_col]].copy()
            temp_df.columns = ['DRUG_NAME', 'TOTAL_SPENDING']
            temp_df['YEAR'] = year

            # Clean records
            temp_df = temp_df.dropna(subset=['DRUG_NAME'])
            temp_df = temp_df[~temp_df['DRUG_NAME'].astype(str).str.contains('The name of the drug|A term referring|Annual Growth|Note:|Description|Brand Name|Generic Name|Manufacturer|Total Spending', case=False, na=False)]
            all_data.append(temp_df)

    if not all_data:
        return pd.DataFrame(columns=['DRUG_NAME', 'TOTAL_SPENDING', 'YEAR'])
    return pd.concat(all_data, ignore_index=True)

cms_raw = extract_cms_spending()
print(f"CMS data loaded from clean Excel worksheets. Rows: {len(cms_raw)}, unique years: {sorted(cms_raw['YEAR'].unique()) if len(cms_raw) > 0 else []}")

Year: 2016 | File: DSD_PTD_R17_DYT16_Web.xlsx | Identified Columns: Name: BRAND NAME, Spending: TOTAL SPENDING
Year: 2017 | File: DSD_PTD_R18_DYT17_Web.xlsx | Identified Columns: Name: BRAND NAME, Spending: TOTAL SPENDING
Year: 2018 | File: DSD_PTD_R19_DYT18_Web.xlsx | Identified Columns: Name: BRAND NAME, Spending: TOTAL SPENDING
Year: 2019 | File: DSD_PTD_R20_DYT19_Web.xlsx | Identified Columns: Name: BRAND NAME, Spending: TOTAL SPENDING
Year: 2020 | File: DSD_PTD_R21_DYT20_Web - 211208.xlsx | Identified Columns: Name: BRAND NAME, Spending: TOTAL SPENDING
Year: 2021 | File: DSD_PTD_R22_DYT21_Web - 230110.xlsx | Identified Columns: Name: BRAND NAME, Spending: TOTAL SPENDING
Year: 2022 | File: DSD_PTD_RY24_DYT22_Web - 240208.xlsx | Identified Columns: Name: BRAND NAME, Spending: TOTAL SPENDING
Year: 2023 | File: DSD_PTD_RY25_DYT23_Web - 250415.xlsx | Identified Columns: Name: BRAND NAME, Spending: TOTAL SPENDING
CMS data loaded from clean Excel worksheets. Rows: 27032, unique years: [n

In [31]:
import pandas as pd
# Inspect the sheets of a representative CMS file
filepath = '/content/DSD_PTD_R22_DYT21_Web - 230110.xlsx'
xl = pd.ExcelFile(filepath)
print("Sheet names in target file:", xl.sheet_names)

Sheet names in target file: ['Data Dictionary', 'Spending & Utilization YTD 2021', 'Manufacturer Summary', 'Drug Use Information']


In [32]:
import pandas as pd
# Let's inspect the sheets of a representative CMS file
filepath = '/content/DSD_PTD_R22_DYT21_Web - 230110.xlsx'
xl = pd.ExcelFile(filepath)
print("Sheet names in target file:", xl.sheet_names)

Sheet names in target file: ['Data Dictionary', 'Spending & Utilization YTD 2021', 'Manufacturer Summary', 'Drug Use Information']


In [44]:
# --- 3. Run Matching, Chronological Splitting, and Model Performance ---
try:
    fda_df_local = fda_df
except NameError:
    fda_df_local, patent_df, excl_df = load_fda_data()

# Standardize CMS names to avoid matching row descriptions
cms_clean = cms_raw.dropna(subset=['DRUG_NAME', 'TOTAL_SPENDING']).copy()
cms_clean['TOTAL_SPENDING'] = pd.to_numeric(cms_clean['TOTAL_SPENDING'], errors='coerce')
cms_clean = cms_clean.dropna(subset=['TOTAL_SPENDING'])
cms_clean = cms_clean[~cms_clean['DRUG_NAME'].str.contains('The name of the drug|A term referring|Annual Growth|Note:|Description|Brand Name|Generic Name|Manufacturer|Total Spending', case=False, na=False)]

fda_names = fda_df_local['DRUG_NAME'].dropna().unique().tolist()
cms_names = cms_clean['DRUG_NAME'].dropna().unique()

matches = {}
exact_cnt = 0
fuzzy_cnt = 0
for name in cms_names:
    cleaned = str(name).strip().upper()
    if cleaned in fda_names:
        matches[name] = cleaned
        exact_cnt += 1
    else:
        close = get_close_matches(cleaned, fda_names, n=1, cutoff=0.70)
        if close:
            matches[name] = close[0]
            fuzzy_cnt += 1

print(f"Matching completed. Exact matches: {exact_cnt}, Fuzzy matches: {fuzzy_cnt}")

cms_clean['MATCHED_NAME'] = cms_clean['DRUG_NAME'].map(matches).fillna('NOMATCH').astype(str)
fda_df_local['DRUG_NAME'] = fda_df_local['DRUG_NAME'].astype(str)

merged_df = pd.merge(cms_clean, fda_df_local, left_on='MATCHED_NAME', right_on='DRUG_NAME', how='inner', suffixes=('_CMS', '_FDA'))

# Sort chronologically to calculate previous year lags correctly
merged_df = merged_df.sort_values(['DRUG_NAME_CMS', 'YEAR'])
merged_df['PREV_YEAR_SPENDING'] = merged_df.groupby('DRUG_NAME_CMS')['TOTAL_SPENDING'].shift(1).fillna(0)

# Features
merged_df['DAYS_TO_EXPIRY'] = (merged_df['PATENT_EXPIRE_DATE'] - pd.to_datetime(merged_df['YEAR'], format='%Y')).dt.days.fillna(9999)
merged_df['DAYS_TO_EXCL'] = (merged_df['EXCLUSIVITY_DATE'] - pd.to_datetime(merged_df['YEAR'], format='%Y')).dt.days.fillna(9999)

# Split
train_data = merged_df[merged_df['YEAR'] < 2023].dropna(subset=['TOTAL_SPENDING'])
test_data = merged_df[merged_df['YEAR'] == 2023].dropna(subset=['TOTAL_SPENDING'])

X_cols = ['PREV_YEAR_SPENDING', 'DAYS_TO_EXPIRY', 'DAYS_TO_EXCL']
y_col = 'TOTAL_SPENDING'

print(f"Train observations: {len(train_data)}, Test observations: {len(test_data)}")

if len(train_data) > 0 and len(test_data) > 0:
    # Train Models
    models = {
        'Ridge': Ridge(),
        'RandomForest': RandomForestRegressor(random_state=42),
        'GradientBoosting': GradientBoostingRegressor(random_state=42)
    }

    for name, m in models.items():
        m.fit(train_data[X_cols], train_data[y_col])
        preds = m.predict(test_data[X_cols])
        mae = mean_absolute_error(test_data[y_col], preds)
        rmse = np.sqrt(mean_squared_error(test_data[y_col], preds))
        r2 = r2_score(test_data[y_col], preds)
        print(f"{name} Test Evaluation -> MAE: ${mae:,.2f}, RMSE: ${rmse:,.2f}, R2: {r2:.4f}")
else:
    print("Warning: Insufficient matched records available in split partitions for regression modeling.")

Matching completed. Exact matches: 3276, Fuzzy matches: 1511
Train observations: 251381, Test observations: 39549
Ridge Test Evaluation -> MAE: $2,218,433.86, RMSE: $19,925,921.71, R2: 0.9860
RandomForest Test Evaluation -> MAE: $5,476,491.36, RMSE: $37,074,001.91, R2: 0.9515
GradientBoosting Test Evaluation -> MAE: $2,970,877.60, RMSE: $33,859,007.52, R2: 0.9595


In [39]:
import pandas as pd
filepath_2023 = '/content/DSD_PTD_RY25_DYT23_Web - 250415.xlsx'
xl = pd.ExcelFile(filepath_2023)
print("2023 Sheets:", xl.sheet_names)

# Read a few rows of the main data sheet
sheet_name = [s for s in xl.sheet_names if 'Spending' in s or 'SPENDING' in s][0]
df_23_raw = pd.read_excel(filepath_2023, sheet_name=sheet_name, nrows=15)
print("\nFirst 15 rows of 2023 spending sheet:")
print(df_23_raw.head(15))

2023 Sheets: ['Data Dictionary', 'Spending & Utilization YTD 2023', 'Manufacturer Summary', 'Drug Use Information']

First 15 rows of 2023 spending sheet:
   Medicare Part D Drug Spending and Utilization, Calendar Years 2019 - 2023  \
0   *Average spending per dosage unit reflects mul...                          
1                                                 NaN                          
2                                          Brand Name                          
3                            1st Tier Unifine Pentips                          
4                       1st Tier Unifine Pentips Plus                          
5                                            Abacavir                          
6                                 Abacavir-Lamivudine                          
7                                             Abilify                          
8                                   Abilify Asimtufii                          
9                                    Abilify 

In [35]:
# Diagnostics: inspect year and spending distributions inside merged_df
print("Observations per year in merged_df:")
print(merged_df['YEAR'].value_counts())

print("\nChecking if 2023 records exist in merged_df:")
print(merged_df[merged_df['YEAR'] == 2023].head(3))

Observations per year in merged_df:
YEAR
2016    31795
Name: count, dtype: int64

Checking if 2023 records exist in merged_df:
Empty DataFrame
Columns: [DRUG_NAME_CMS, TOTAL_SPENDING, YEAR, MATCHED_NAME, INGREDIENT, DF;ROUTE, TRADE_NAME, APPLICANT, STRENGTH, APPL_TYPE, APPL_NO, PRODUCT_NO, TE_CODE, APPROVAL_DATE, RLD, RS, TYPE, APPLICANT_FULL_NAME, PATENT_EXPIRE_DATE, EXCLUSIVITY_DATE, DRUG_NAME_FDA, PREV_YEAR_SPENDING, DAYS_TO_EXPIRY, DAYS_TO_EXCL]
Index: []

[0 rows x 24 columns]


In [36]:
# Diagnostics: inspect year and spending distributions inside merged_df
print("Observations per year in merged_df:")
print(merged_df['YEAR'].value_counts())

print("\nChecking if 2023 records exist in merged_df:")
print(merged_df[merged_df['YEAR'] == 2023].head(3))

Observations per year in merged_df:
YEAR
2016    31795
Name: count, dtype: int64

Checking if 2023 records exist in merged_df:
Empty DataFrame
Columns: [DRUG_NAME_CMS, TOTAL_SPENDING, YEAR, MATCHED_NAME, INGREDIENT, DF;ROUTE, TRADE_NAME, APPLICANT, STRENGTH, APPL_TYPE, APPL_NO, PRODUCT_NO, TE_CODE, APPROVAL_DATE, RLD, RS, TYPE, APPLICANT_FULL_NAME, PATENT_EXPIRE_DATE, EXCLUSIVITY_DATE, DRUG_NAME_FDA, PREV_YEAR_SPENDING, DAYS_TO_EXPIRY, DAYS_TO_EXCL]
Index: []

[0 rows x 24 columns]


In [12]:
import pandas as pd

for f_name in ['patent.txt', 'exclusivity.txt', 'products.txt']:
    path = f'/content/{f_name}'
    df_sample = pd.read_csv(path, sep='\t', nrows=5, encoding='latin1')
    print(f'=== File: {f_name} ===')
    print('Original Columns:', list(df_sample.columns))
    print(df_sample.head(2))

=== File: patent.txt ===
Original Columns: ['Appl_Type~Appl_No~Product_No~Patent_No~Patent_Expire_Date_Text~Drug_Substance_Flag~Drug_Product_Flag~Patent_Use_Code~Delist_Flag~Submission_Date']
  Appl_Type~Appl_No~Product_No~Patent_No~Patent_Expire_Date_Text~Drug_Substance_Flag~Drug_Product_Flag~Patent_Use_Code~Delist_Flag~Submission_Date
0        N~020610~001~7625884~Aug 24, 2026~~~U-141~~                                                                                              
1       N~018613~001~7560445~Feb 1, 2027~Y~Y~U-986~~                                                                                              
=== File: exclusivity.txt ===
Original Columns: ['Appl_Type~Appl_No~Product_No~Exclusivity_Code~Exclusivity_Date']
  Appl_Type~Appl_No~Product_No~Exclusivity_Code~Exclusivity_Date
0                      N~017031~001~RTO~Jul 13, 2026            
1                    N~018680~001~D-193~Jun 28, 2027            
=== File: products.txt ===
Original Columns: ['Ingredient

In [13]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
from difflib import get_close_matches
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- 1. Load and Parse FDA Patent/Exclusivity Data using Tilde (~) ---
def load_fda_data():
    patent_path = '/content/patent.txt'
    excl_path = '/content/exclusivity.txt'
    prod_path = '/content/products.txt'

    # Use correct separator '~' based on inspection results
    patent_df = pd.read_csv(patent_path, sep='~', on_bad_lines='skip', encoding='latin1')
    excl_df = pd.read_csv(excl_path, sep='~', on_bad_lines='skip', encoding='latin1')
    prod_df = pd.read_csv(prod_path, sep='~', on_bad_lines='skip', encoding='latin1')

    # Normalize columns
    for df in [patent_df, excl_df, prod_df]:
        df.columns = df.columns.str.strip().str.upper()

    # Map Patent_Expire_Date_Text to Patent_Expire_Date
    patent_df['PATENT_EXPIRE_DATE'] = pd.to_datetime(patent_df['PATENT_EXPIRE_DATE_TEXT'], errors='coerce')
    excl_df['EXCLUSIVITY_DATE'] = pd.to_datetime(excl_df['EXCLUSIVITY_DATE'], errors='coerce')

    # Track valid counts
    valid_patents = patent_df['PATENT_EXPIRE_DATE'].notnull().sum()
    valid_excl = excl_df['EXCLUSIVITY_DATE'].notnull().sum()

    # Deduplicate patents and exclusivity per Application Number (earliest date first)
    pat_dedup = patent_df.dropna(subset=['PATENT_EXPIRE_DATE']).sort_values('PATENT_EXPIRE_DATE').groupby('APPL_NO').first().reset_index()
    excl_dedup = excl_df.dropna(subset=['EXCLUSIVITY_DATE']).sort_values('EXCLUSIVITY_DATE').groupby('APPL_NO').first().reset_index()

    fda_merged = pd.merge(prod_df, pat_dedup[['APPL_NO', 'PATENT_EXPIRE_DATE']], on='APPL_NO', how='left')
    fda_merged = pd.merge(fda_merged, excl_dedup[['APPL_NO', 'EXCLUSIVITY_DATE']], on='APPL_NO', how='left')

    # Use Trade_Name as DRUG_NAME to match CMS drug naming format
    fda_merged['DRUG_NAME'] = fda_merged['TRADE_NAME'].str.strip().str.upper()

    return fda_merged, patent_df, excl_df

fda_df, patent_df, excl_df = load_fda_data()
print(f"FDA records parsed correctly. Valid patent dates: {patent_df['PATENT_EXPIRE_DATE'].notnull().sum()}, Valid exclusivity dates: {excl_df['EXCLUSIVITY_DATE'].notnull().sum()}")


FDA records parsed correctly. Valid patent dates: 21112, Valid exclusivity dates: 2110


In [20]:
# --- 2. Clean CMS and run match coverage ---
# Standardize CMS names to avoid matching row descriptions
cms_clean = cms_raw.dropna(subset=['DRUG_NAME', 'TOTAL_SPENDING']).copy()

# Convert TOTAL_SPENDING to numeric, forcing non-numeric headers to NaN, then drop them
cms_clean['TOTAL_SPENDING'] = pd.to_numeric(cms_clean['TOTAL_SPENDING'], errors='coerce')
cms_clean = cms_clean.dropna(subset=['TOTAL_SPENDING'])

# Filter out systematic metadata strings in drug names
cms_clean = cms_clean[~cms_clean['DRUG_NAME'].str.contains('The name of the drug|A term referring|Annual Growth|Note:|Description|Brand Name|Generic Name|Manufacturer|Total Spending', case=False, na=False)]

fda_names = fda_df['DRUG_NAME'].dropna().unique().tolist()
cms_names = cms_clean['DRUG_NAME'].dropna().unique()

matches = {}
exact_cnt = 0
fuzzy_cnt = 0
for name in cms_names:
    cleaned = str(name).strip().upper()
    # Simple direct matching check
    if cleaned in fda_names:
        matches[name] = cleaned
        exact_cnt += 1
    else:
        # Fuzzy match
        close = get_close_matches(cleaned, fda_names, n=1, cutoff=0.70) # Lowered cutoff for robust matching
        if close:
            matches[name] = close[0]
            fuzzy_cnt += 1

print(f"Product Matching -> Exact Matches: {exact_cnt}, Fuzzy Matches: {fuzzy_cnt}, Unmatched: {len(cms_names) - (exact_cnt + fuzzy_cnt)}")

Product Matching -> Exact Matches: 0, Fuzzy Matches: 0, Unmatched: 0


In [17]:
print(cms_raw.dropna(subset=['DRUG_NAME']).head(20))

                                            DRUG_NAME  \
1                                         Description   
2   The name of the drug filled. This includes bot...   
3   A term referring to the chemical ingredient of...   
4               Name of the manufacturer of the drug.   
5              Number of manufacturers for each drug.   
6   Aggregate drug spending for the Part D program...   
7   The sum of the dosage units of medication disp...   
8   Number of prescription fills for each drug. In...   
9   Number of Medicare Part D beneficiaries utiliz...   
10  Part D drug spending divided by the number of ...   
11  Total Part D drug spending divided by the numb...   
12  Total Part D drug spending divided by the numb...   
13  The percent change in average spending per dos...   
14  The constant average change in spending per do...   
16                                        Description   
17  The name of the drug filled. This includes bot...   
18  A term referring to the che

In [18]:
print(cms_raw.dropna(subset=['DRUG_NAME']).head(20))

                                            DRUG_NAME  \
1                                         Description   
2   The name of the drug filled. This includes bot...   
3   A term referring to the chemical ingredient of...   
4               Name of the manufacturer of the drug.   
5              Number of manufacturers for each drug.   
6   Aggregate drug spending for the Part D program...   
7   The sum of the dosage units of medication disp...   
8   Number of prescription fills for each drug. In...   
9   Number of Medicare Part D beneficiaries utiliz...   
10  Part D drug spending divided by the number of ...   
11  Total Part D drug spending divided by the numb...   
12  Total Part D drug spending divided by the numb...   
13  The percent change in average spending per dos...   
14  The constant average change in spending per do...   
16                                        Description   
17  The name of the drug filled. This includes bot...   
18  A term referring to the che

In [21]:
# --- 3. Build Features, Train Models and Perform Ablation ---
cms_clean['MATCHED_NAME'] = cms_clean['DRUG_NAME'].map(matches).astype(str)
fda_df['DRUG_NAME'] = fda_df['DRUG_NAME'].astype(str)

# Safe Merge
merged_df = pd.merge(cms_clean, fda_df, left_on='MATCHED_NAME', right_on='DRUG_NAME', how='left', suffixes=('_CMS', '_FDA'))

# Group and compute chronological lags per product-year
merged_df = merged_df.sort_values(['DRUG_NAME_CMS', 'YEAR'])
merged_df['PREV_YEAR_SPENDING'] = merged_df.groupby('DRUG_NAME_CMS')['TOTAL_SPENDING'].shift(1).fillna(0)

# Numeric features for remaining timelines
merged_df['DAYS_TO_EXPIRY'] = (merged_df['PATENT_EXPIRE_DATE'] - pd.to_datetime(merged_df['YEAR'], format='%Y')).dt.days.fillna(9999)
merged_df['DAYS_TO_EXCL'] = (merged_df['EXCLUSIVITY_DATE'] - pd.to_datetime(merged_df['YEAR'], format='%Y')).dt.days.fillna(9999)

train_data = merged_df[merged_df['YEAR'] < 2023]
test_data = merged_df[merged_df['YEAR'] == 2023]

X_cols = ['PREV_YEAR_SPENDING', 'DAYS_TO_EXPIRY', 'DAYS_TO_EXCL']
y_col = 'TOTAL_SPENDING'

print(f"Train observations: {len(train_data)}, Test observations: {len(test_data)}")

# Execute model evaluations if test data contains records
if len(test_data) > 0 and len(train_data) > 0:
    models = {
        'Ridge': Ridge(),
        'RandomForest': RandomForestRegressor(random_state=42),
        'GradientBoosting': GradientBoostingRegressor(random_state=42)
    }

    for name, m in models.items():
        m.fit(train_data[X_cols], train_data[y_col])
        preds = m.predict(test_data[X_cols])
        mae = mean_absolute_error(test_data[y_col], preds)
        rmse = mean_squared_error(test_data[y_col], preds, squared=False)
        r2 = r2_score(test_data[y_col], preds)
        print(f"{name} Model -> MAE: ${mae:,.2f}, RMSE: ${rmse:,.2f}, R2: {r2:.4f}")
else:
    print("Warning: Insufficient matched records available in split partitions.")

Train observations: 0, Test observations: 0
